In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Input

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import GRU
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau

2026-05-07 14:29:35.102044: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778164175.607600      76 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778164175.814889      76 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778164177.048028      76 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778164177.048077      76 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778164177.048079      76 computation_placer.cc:177] computation placer alr

In [2]:
data = pd.read_csv("/kaggle/input/datasets/nimishagrawal17/electric-production/Electric_Production.csv")

In [3]:
data['Value'] = pd.to_numeric(data['Value'])

values = data['Value'].values.reshape(-1,1)

In [4]:
scaler = MinMaxScaler()

scaled_data = scaler.fit_transform(values)

In [5]:
sequence_length = 24

x = []
y = []

for i in range(sequence_length,len(scaled_data)):
    
    x.append(scaled_data[i-sequence_length:i])
    y.append(scaled_data[i])

x = np.array(x)
y = np.array(y)


In [6]:
train_size = int(len(x)*0.8)

x_train = x[:train_size]
y_train = y[:train_size]

x_test = x[train_size:]
y_test = y[train_size:]



In [7]:
model = Sequential()

model.add(
    GRU(
        128,
        return_sequences=True,
        input_shape=(x_train.shape[1],1)
    )
)

model.add(Dropout(0.2))

model.add(
    GRU(
        64,
        return_sequences=False
    )
)

model.add(Dropout(0.2))

model.add(Dense(32,activation='relu'))

model.add(Dense(1))

2026-05-07 14:30:17.208304: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [8]:
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

In [9]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3
)

In [10]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_test,y_test),
    epochs=150,
    batch_size=8,
    callbacks=[early_stop,reduce_lr]
)


Epoch 1/150
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - loss: 0.0633 - mae: 0.1935 - val_loss: 0.0449 - val_mae: 0.1690 - learning_rate: 0.0010
Epoch 2/150
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.0181 - mae: 0.1084 - val_loss: 0.0220 - val_mae: 0.1220 - learning_rate: 0.0010
Epoch 3/150
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0185 - mae: 0.1140 - val_loss: 0.0268 - val_mae: 0.1325 - learning_rate: 0.0010
Epoch 4/150
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0166 - mae: 0.1049 - val_loss: 0.0231 - val_mae: 0.1239 - learning_rate: 0.0010
Epoch 5/150
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0149 - mae: 0.1002 - val_loss: 0.0185 - val_mae: 0.1136 - learning_rate: 0.0010
Epoch 6/150
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0135 - mae: 0.0978 - val_loss: 0.0235 - val_mae: 0.1239 - learning_rate: 0.0010
Epoch 7/150
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0172 - mae: 0.1084 - val_loss: 0.0185 - val_mae: 0.1110 - learning_rate: 0.0010
Epoch 

In [11]:
predictions = model.predict(x_test)

predictions = scaler.inverse_transform(predictions)

y_test_actual = scaler.inverse_transform(y_test)

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 132ms/step


In [12]:
rmse = np.sqrt(
    mean_squared_error(
        y_test_actual,
        predictions
    )
)

mae = mean_absolute_error(
    y_test_actual,
    predictions
)

r2 = r2_score(
    y_test_actual,
    predictions
)

accuracy = r2 * 100


In [13]:
print("RMSE :",rmse)
print("MAE :",mae)
print("R2 Score :",r2)
print("Accuracy :",accuracy)

RMSE : 4.820975467753744
MAE : 3.64112173486328
R2 Score : 0.7471016534728133
Accuracy : 74.71016534728133
